# Task Dataset Build

Species-detector dataset: target species (XC/eBird) plus other-present species and AudioSet ambient (latter two pooled into `non_target`). Downloads are cached.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path

import pyrootutils

ROOT = pyrootutils.setup_root(
    search_from=Path.cwd(),
    indicator="pyproject.toml",
    pythonpath=True,
    dotenv=True,
)

# Parameters

In [21]:
from building.polychirp_training import task_slug

LAT, LON = 39.305613, -76.588305 # greater bmore area
RADIUS_KM = 500

TARGET_SPECIES = [
    "Anas platyrhynchos",
    "Coragyps atratus",
    "Colaptes auratus",
    "Cyanocitta cristata",
    "Corvus brachyrhynchos",
    "Poecile carolinensis",
    "Baeolophus bicolor",
    "Thryothorus ludovicianus",
    "Sturnus vulgaris",
    "Turdus migratorius",
    "Passer domesticus",
    "Melospiza melodia",
    "Agelaius phoeniceus",
    "Quiscalus quiscula",
    "Cardinalis cardinalis"
]

TARGET_CLIPS_PER_SPECIES = 5000
NON_TARGET_CLIPS_PER_SPECIES = 300

# Area discovery (matches 00_area_analysis.ipynb defaults)
MIN_AREA_RECORDINGS = 10
ANALYSIS_MAX_BIRDNET = 500
ANALYSIS_MIN_CONF = 0.85

# AudioSet ambient noise across about 60 FOCUS_CLASSES
AS_CLIPS_PER_CLASS = 250
AS_MAX_TOTAL_CLIPS = 15000

# Slug-named folders keep the (species, area) context visible on disk.
COLLECTION_NAME = task_slug(TARGET_SPECIES, LAT, LON, RADIUS_KM)
print(COLLECTION_NAME)

task_s15_39_-77_r500


In [5]:
%pip install -e

Note: you may need to restart the kernel to use updated packages.


C:\Users\annab\Downloads\EDL\birdshitterReal\EDL-PolyChirp\.venv\Scripts\python.exe: No module named pip


In [9]:
import os
from dotenv import load_dotenv

# Force load the .env file (specify path if it's in a custom location)
load_dotenv()

api_key = os.getenv("XC_API_KEY")
print("Loaded API Key:", api_key[:6] + "..." if api_key else "NOT FOUND")

Loaded API Key: 351489...


In [22]:
from building.analysis import bbox_from_radius, fetch_area_recordings

bbox = bbox_from_radius(39.305613, -76.588305, 500)
recs = await fetch_area_recordings(bbox)
print("Number of raw recordings fetched:", len(recs))

Number of raw recordings fetched: 2973


In [18]:
from building.analysis import (
    area_audio_cache_dir,
    bbox_from_radius,
    birdnet_area_analysis,
    combined_species_table,
    fetch_area_recordings,
    recordings_to_df,
)

bbox = bbox_from_radius(LAT, LON, RADIUS_KM)
area_df = recordings_to_df(await fetch_area_recordings(bbox))

print("Columns in area_df:", area_df.columns.tolist())
display(area_df.head(2))

Columns in area_df: ['id', 'scientific_name', 'gen', 'sp', 'en', 'lat', 'lon', 'q', 'length_seconds', 'cnt', 'loc', 'date', 'also', 'annotation_set', 'file']


,id,scientific_name,gen,sp,en,lat,lon,q,length_seconds,cnt,loc,date,also,annotation_set,file
0,140021,Anser caerulescens,Anser,caerulescens,Snow Goose,39.2518,-75.4527,A,77,United States,"Bombay Hook National Wildlife Refuge, Smyrna, ...",2012-02-16,[],None,https://xeno-canto.org/140021/download
1,1185239,Antrostomus vociferus,Antrostomus,vociferus,Eastern Whip-poor-will,38.8821,-77.6891,A,79,United States,VIRGINIA: Prince William County; Bull Run Moun...,2020-05-15,[],None,https://xeno-canto.org/1185239/download


# Discover other species present in the area

In [23]:
from building.analysis import (
    area_audio_cache_dir,
    bbox_from_radius,
    birdnet_area_analysis,
    combined_species_table,
    fetch_area_recordings,
    recordings_to_df,
)

bbox = bbox_from_radius(LAT, LON, RADIUS_KM)
area_df = recordings_to_df(await fetch_area_recordings(bbox))

bn = await birdnet_area_analysis(
    area_df,
    max_recordings=ANALYSIS_MAX_BIRDNET,
    min_confidence=ANALYSIS_MIN_CONF,
    cache_dir=area_audio_cache_dir(LAT, LON, RADIUS_KM),
)
combined = combined_species_table(
    area_df, bn["detections"], min_recordings=MIN_AREA_RECORDINGS
)
OTHER_SPECIES = [
    sp for sp in combined["scientific_name"].tolist() if sp not in TARGET_SPECIES
]
print(f"{len(OTHER_SPECIES)} other species present in area")
OTHER_SPECIES

BirdNET analyse:   0%|          | 0/500 [00:00<?, ?it/s]

111 other species present in area


['Seiurus aurocapilla',
 'Geothlypis trichas',
 'Pipilo erythrophthalmus',
 'Vireo olivaceus',
 'Dumetella carolinensis',
 'Poecile atricapillus',
 'Spinus tristis',
 'Hylocichla mustelina',
 'Setophaga aestiva',
 'Setophaga pinus',
 'Setophaga ruticilla',
 'Setophaga pensylvanica',
 'Spizella passerina',
 'Passerina cyanea',
 'Piranga olivacea',
 'Mniotilta varia',
 'Melanerpes carolinus',
 'Setophaga citrina',
 'Spizella pusilla',
 'Melospiza georgiana',
 'Icterus galbula',
 'Catharus fuscescens',
 'Sitta carolinensis',
 'Troglodytes aedon',
 'Zenaida macroura',
 'Setophaga caerulescens',
 'Zonotrichia albicollis',
 'Branta canadensis',
 'Dryobates pubescens',
 'Contopus virens',
 'Vireo solitarius',
 'Parkesia motacilla',
 'Setophaga discolor',
 'Setophaga virens',
 'Catharus guttatus',
 'Setophaga fusca',
 'Helmitheros vermivorum',
 'Myiarchus crinitus',
 'Troglodytes hiemalis',
 'Mimus polyglottos',
 'Pheucticus ludovicianus',
 'Setophaga coronata',
 'Cardellina canadensis',
 'Say

# Stream-download target species

In [24]:
from building.data.download import download_and_process
from building.data.sources import XCSource, EBirdSource

with XCSource() as xc, EBirdSource() as eb:
    for sp in TARGET_SPECIES:
        await download_and_process(sp, TARGET_CLIPS_PER_SPECIES, [xc, eb])


=== [Anas platyrhynchos] target=10000 clips (5000/source x 2 sources)  on disk=24 [xc=24/5000, ebird=0/5000] ===
[Anas platyrhynchos] --- Phase A: per-source quota (parallel) ---
[Anas platyrhynchos / xc] 24/5000 on disk; queued 126 of 136 available.


[Anas platyrhynchos/xc] recs:   0%|          | 0/126 [00:00<?, ?it/s]

[Anas platyrhynchos/xc] clips:   0%|          | 24/5000 [00:00<?, ?it/s]

[Anas platyrhynchos / ebird] Phase A failed: SSLError: HTTPSConnectionPool(host='media.ebird.org', port=443): Max retries exceeded with url: /api/v1/search?taxonCode=mallar3&mediaType=audio&count=100&sort=rating_rank_desc (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1010)')))
[Anas platyrhynchos] --- Phase B: top up (have 369/10000) ---
[Anas platyrhynchos / xc] no pending recordings (available=136, already processed=136, on disk=369/10000).
[Anas platyrhynchos / ebird] Phase B failed: SSLError: HTTPSConnectionPool(host='media.ebird.org', port=443): Max retries exceeded with url: /api/v1/search?taxonCode=mallar3&mediaType=audio&count=100&sort=rating_rank_desc (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1010)')))
[Anas platyrhynchos] done. final 369/10000 clips on dis

[Coragyps atratus/xc] recs:   0%|          | 0/26 [00:00<?, ?it/s]

...

[Quiscalus quiscula/xc] clips:   0%|          | 0/5000 [00:00<?, ?it/s]

[Quiscalus quiscula] --- Phase B: top up (have 533/10000) ---
[Quiscalus quiscula / xc] no pending recordings (available=92, already processed=92, on disk=533/10000).
[Quiscalus quiscula / ebird] Phase B failed: SSLError: HTTPSConnectionPool(host='media.ebird.org', port=443): Max retries exceeded with url: /api/v1/search?taxonCode=comgra&mediaType=audio&count=100&sort=rating_rank_desc (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1010)')))
[Quiscalus quiscula] done. final 533/10000 clips on disk [xc=533, ebird=0].

=== [Cardinalis cardinalis] target=10000 clips (5000/source x 2 sources)  on disk=0 [xc=0/5000, ebird=0/5000] ===
[Cardinalis cardinalis] --- Phase A: per-source quota (parallel) ---
[Cardinalis cardinalis / ebird] Phase A failed: SSLError: HTTPSConnectionPool(host='media.ebird.org', port=443): Max retries exceeded with url: /api/v1/search?taxonCode=norcar&mediaType=

[Cardinalis cardinalis/xc] recs:   0%|          | 0/364 [00:00<?, ?it/s]

[Cardinalis cardinalis/xc] clips:   0%|          | 0/5000 [00:00<?, ?it/s]

[Cardinalis cardinalis] --- Phase B: top up (have 1078/10000) ---
[Cardinalis cardinalis / xc] no pending recordings (available=364, already processed=364, on disk=1078/10000).
[Cardinalis cardinalis / ebird] Phase B failed: SSLError: HTTPSConnectionPool(host='media.ebird.org', port=443): Max retries exceeded with url: /api/v1/search?taxonCode=norcar&mediaType=audio&count=100&sort=rating_rank_desc (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1010)')))
[Cardinalis cardinalis] done. final 1078/10000 clips on disk [xc=1078, ebird=0].


# Stream-download other species (250 each, pooled as non-target)

In [25]:
with XCSource() as xc, EBirdSource() as eb:
    for sp in OTHER_SPECIES:
        await download_and_process(sp, NON_TARGET_CLIPS_PER_SPECIES, [xc, eb])


=== [Seiurus aurocapilla] target=600 clips (300/source x 2 sources)  on disk=0 [xc=0/300, ebird=0/300] ===
[Seiurus aurocapilla] --- Phase A: per-source quota (parallel) ---
[Seiurus aurocapilla / ebird] Phase A failed: SSLError: HTTPSConnectionPool(host='media.ebird.org', port=443): Max retries exceeded with url: /api/v1/search?taxonCode=ovenbi1&mediaType=audio&count=100&sort=rating_rank_desc (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1010)')))
[Seiurus aurocapilla / xc] 0/300 on disk; queued 182 of 182 available.


[Seiurus aurocapilla/xc] recs:   0%|          | 0/182 [00:00<?, ?it/s]

[Seiurus aurocapilla/xc] clips:   0%|          | 0/300 [00:00<?, ?it/s]

[Seiurus aurocapilla / xc] reached 300 clips.
[Seiurus aurocapilla] --- Phase B: top up (have 300/600) ---
[Seiurus aurocapilla / xc] 300/600 on disk; queued 109 of 182 available.


[Seiurus aurocapilla/xc] recs:   0%|          | 0/109 [00:00<?, ?it/s]

...

[Tringa melanoleuca/xc] clips:   0%|          | 0/300 [00:00<?, ?it/s]

[Tringa melanoleuca / xc] reached 300 clips.
[Tringa melanoleuca] --- Phase B: top up (have 300/600) ---
[Tringa melanoleuca / xc] 300/600 on disk; queued 13 of 94 available.


[Tringa melanoleuca/xc] recs:   0%|          | 0/13 [00:00<?, ?it/s]

[Tringa melanoleuca/xc] clips:  50%|#####     | 300/600 [00:00<?, ?it/s]

[Tringa melanoleuca / ebird] Phase B failed: SSLError: HTTPSConnectionPool(host='media.ebird.org', port=443): Max retries exceeded with url: /api/v1/search?taxonCode=greyel&mediaType=audio&count=100&sort=rating_rank_desc (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1010)')))
[Tringa melanoleuca] done. final 349/600 clips on disk [xc=349, ebird=0].

=== [Dryobates villosus] target=600 clips (300/source x 2 sources)  on disk=0 [xc=0/300, ebird=0/300] ===
[Dryobates villosus] --- Phase A: per-source quota (parallel) ---
[Dryobates villosus / ebird] Phase A failed: KeyError: "No eBird species code for 'Dryobates villosus'"
[Dryobates villosus / xc] no pending recordings (available=0, already processed=0, on disk=0/300).
[Dryobates villosus] --- Phase B: top up (have 0/600) ---
[Dryobates villosus / xc] no pending recordings (available=0, already processed=0, on disk=0/600).
[Dryob

# Boost `no_bird` via OTHER_SPECIES reprocessing

BirdNET mines a 3 s window into `non_target_empty/` from recordings with no detection >= `BIRDNET_THRESHOLD` (0.92). Each pass doubles the `OTHER_SPECIES` quota; stops at `NO_BIRD_CAP` or when sources run out.

In [31]:
from building.data.download import (
    BIRDNET_NO_BIRD_DIR, NO_BIRD_CAP, SUBSAMPLES_DIR, download_and_process,
)
from building.data.sources import XCSource, EBirdSource

NO_BIRD_TARGET = NO_BIRD_CAP   # 20_000


def n_no_bird() -> int:
    return sum(1 for _ in BIRDNET_NO_BIRD_DIR.glob("*.wav")) if BIRDNET_NO_BIRD_DIR.exists() else 0


def total_other_clips() -> int:
    return sum(
        sum(1 for _ in (SUBSAMPLES_DIR / sp.replace(" ", "_")).glob("*.wav"))
        for sp in OTHER_SPECIES
        if (SUBSAMPLES_DIR / sp.replace(" ", "_")).exists()
    )


curr_nb, curr_ot = n_no_bird(), total_other_clips()
print(f"start  no_bird={curr_nb}/{NO_BIRD_TARGET}  other_total={curr_ot}")

quota = NON_TARGET_CLIPS_PER_SPECIES
done = False
while not done and curr_nb < NO_BIRD_TARGET:
    quota *= 2
    print(f"\n>>> boost OTHER_SPECIES quota -> {quota}/source")
    with XCSource() as xc, EBirdSource() as eb:
        for sp in OTHER_SPECIES:
            before = n_no_bird()
            #await download_and_process(sp, quota, [xc, eb], verbose=False)
            await download_and_process(sp, quota, [xc], verbose=False)
            after = n_no_bird()
            print(f"  {sp:<32s} no_bird {before:>5d} -> {after:>5d} (+{after - before})")
            if after >= NO_BIRD_TARGET:
                done = True
                break
    new_nb, new_ot = n_no_bird(), total_other_clips()
    print(
        f"after pass: no_bird={new_nb} (+{new_nb - curr_nb})  "
        f"other_total={new_ot} (+{new_ot - curr_ot})"
    )
    if new_ot == curr_ot:
        print("no new OTHER_SPECIES clips - sources exhausted, stopping")
        break
    curr_nb, curr_ot = new_nb, new_ot

print(f"\nend    no_bird={curr_nb}/{NO_BIRD_TARGET}")

start  no_bird=170/20000  other_total=49659

>>> boost OTHER_SPECIES quota -> 600/source
  Seiurus aurocapilla              no_bird   170 ->   170 (+0)
  Geothlypis trichas               no_bird   170 ->   170 (+0)
  Pipilo erythrophthalmus          no_bird   170 ->   170 (+0)
  Vireo olivaceus                  no_bird   170 ->   170 (+0)
  Dumetella carolinensis           no_bird   170 ->   170 (+0)
  Poecile atricapillus             no_bird   170 ->   170 (+0)
  Spinus tristis                   no_bird   170 ->   170 (+0)
  Hylocichla mustelina             no_bird   170 ->   170 (+0)
  Setophaga aestiva                no_bird   170 ->   170 (+0)
  Setophaga pinus                  no_bird   170 ->   170 (+0)
  Setophaga ruticilla              no_bird   170 ->   170 (+0)
  Setophaga pensylvanica           no_bird   170 ->   170 (+0)
  Spizella passerina               no_bird   170 ->   170 (+0)
  Passerina cyanea                 no_bird   170 ->   170 (+0)
  Piranga olivacea           

In [27]:
import os
import certifi

# Force Python and requests/urllib3 to use certifi's updated bundle immediately
os.environ['SSL_CERT_FILE'] = certifi.where()
os.environ['REQUESTS_CA_BUNDLE'] = certifi.where()

print("Updated certificate path to:", certifi.where())

Updated certificate path to: C:\Users\annab\Downloads\EDL\EDL-Polychirp\.venv\Lib\site-packages\certifi\cacert.pem


In [28]:
import os
import certifi

# Force the environment to use your virtual environment's certifi bundle
os.environ['SSL_CERT_FILE'] = certifi.where()
os.environ['REQUESTS_CA_BUNDLE'] = certifi.where()

# Test if eBird works now
import requests
try:
    response = requests.get("https://media.ebird.org", timeout=5)
    print(f"Success! eBird responded with status code: {response.status_code}")
except Exception as e:
    print(f"Still failing: {e}")

Success! eBird responded with status code: 200


In [33]:
import os
import shutil
import sys
from pathlib import Path

# 1. Install yt-dlp and imageio-ffmpeg into the active environment
!{sys.executable} -m pip install yt-dlp imageio-ffmpeg

# 2. Locate ffmpeg bundled via imageio-ffmpeg
import imageio_ffmpeg
ffmpeg_src = Path(imageio_ffmpeg.get_ffmpeg_exe())

# Target the virtual environment's Scripts directory (where yt-dlp and python live)
scripts_dir = Path(sys.executable).parent
target_ffmpeg = scripts_dir / ("ffmpeg.exe" if os.name == "nt" else "ffmpeg")

# Copy ffmpeg into the Scripts directory if not already there
if not target_ffmpeg.exists():
    shutil.copy(ffmpeg_src, target_ffmpeg)
    print(f"Copied ffmpeg to: {target_ffmpeg}")

# 3. Dynamically add the Scripts directory to os.environ['PATH']
env_path = os.environ.get("PATH", "")
if str(scripts_dir) not in env_path:
    os.environ["PATH"] = str(scripts_dir) + os.pathsep + env_path

# 4. Verify that shutil.which detects both successfully
print("yt-dlp found at:", shutil.which("yt-dlp"))
print("ffmpeg found at:", shutil.which("ffmpeg"))

   ---------------------------------------- 0.0/3.2 MB ? eta -:--:--
   ------------------- -------------------- 1.6/3.2 MB 8.4 MB/s eta 0:00:01
   ---------------------------------------- 3.2/3.2 MB 8.2 MB/s  0:00:00
   ---------------------------------------- 0.0/31.2 MB ? eta -:--:--
   -- ------------------------------------- 1.8/31.2 MB 9.1 MB/s eta 0:00:04
   ----- ---------------------------------- 4.2/31.2 MB 10.1 MB/s eta 0:00:03
   -------- ------------------------------- 6.3/31.2 MB 10.4 MB/s eta 0:00:03
   ----------- ---------------------------- 8.7/31.2 MB 10.3 MB/s eta 0:00:03
   -------------- ------------------------- 11.5/31.2 MB 11.1 MB/s eta 0:00:02
   ------------------ --------------------- 14.7/31.2 MB 11.7 MB/s eta 0:00:02
   ----------------------- ---------------- 18.1/31.2 MB 12.3 MB/s eta 0:00:02
   --------------------------- ------------ 21.8/31.2 MB 12.9 MB/s eta 0:00:01
   -------------------------------- ------- 25.4/31.2 MB 13.3 MB/s eta 0:00:01
   ---

# Stream-download AudioSet ambient clips

In [ ]:
from building.data.audioset import AudioSetConfig, stream_download_audioset_async

as_cfg = AudioSetConfig(
    clips_per_class=AS_CLIPS_PER_CLASS,
    max_total_clips=AS_MAX_TOTAL_CLIPS,
)
await stream_download_audioset_async(as_cfg)

[audioset] split=unbalanced_train per_class=250 global=15000 on_disk=0 workers=6
[audioset] indexing unbalanced_train segments...
[audioset] candidates per class: Speech=936732, Male speech, man speaking=6941, Female speech, woman speaking=5057, Conversation=2140, Narration, monologue=15253, ...


[Speech] recs:   0%|          | 0/936732 [00:00<?, ?it/s]

[Speech] clips:   0%|          | 0/250 [00:00<?, ?it/s]

# Pre-split summary

In [ ]:
from building.data.dataset import AUDIOSET_DIR, BIRDNET_NO_BIRD_DIR, SUBSAMPLES_DIR


def _count(folder):
    return len(list(folder.glob("*.wav"))) if folder.exists() else 0


print("Target classes:")
for sp in TARGET_SPECIES:
    print(f"  {sp}: {_count(SUBSAMPLES_DIR / sp.replace(' ', '_'))}")

as_total, as_classes = 0, 0
if AUDIOSET_DIR.exists():
    for d in sorted(AUDIOSET_DIR.iterdir()):
        if d.is_dir():
            n = _count(d)
            if n:
                as_total += n
                as_classes += 1
print(f"\nAudioSet: {as_total} clips across {as_classes} classes")

other_total, other_species_present = 0, 0
for sp in OTHER_SPECIES:
    n = _count(SUBSAMPLES_DIR / sp.replace(" ", "_"))
    if n:
        other_total += n
        other_species_present += 1
print(
    f"\nOther birds: {other_total} clips "
    f"across {other_species_present}/{len(OTHER_SPECIES)} species"
)

print(f"\nBirdNET no-bird clips: {_count(BIRDNET_NO_BIRD_DIR)}")

# Per-source RMS distribution (pre-split)

Raw clip levels per source, 2 dB bins, -70..0 dBFS.

In [ ]:
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from building.data.dataset import AUDIOSET_DIR, BIRDNET_NO_BIRD_DIR, SUBSAMPLES_DIR


def _rms_dbfs(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=np.float32)
    rms = np.sqrt(np.mean(x**2))
    return float(20.0 * np.log10(max(rms, 1e-10)))


def _scan(files: list) -> np.ndarray:
    out: list[float] = []
    for f in tqdm(files, desc=f"scan ({len(files)})", leave=False):
        try:
            x, _ = sf.read(str(f), dtype="float32", always_2d=False)
            if x.ndim > 1:
                x = x.mean(axis=-1)
            out.append(_rms_dbfs(x))
        except Exception:
            pass
    return np.array(out)


groups: list[tuple[str, np.ndarray]] = []

# One panel per target species.
for sp in TARGET_SPECIES:
    folder = SUBSAMPLES_DIR / sp.replace(" ", "_")
    if folder.exists():
        groups.append((sp, _scan(sorted(folder.glob("*.wav")))))

# Other birds aggregated.
other_files: list = []
for sp in OTHER_SPECIES:
    folder = SUBSAMPLES_DIR / sp.replace(" ", "_")
    if folder.exists():
        other_files.extend(folder.glob("*.wav"))
groups.append((f"Other birds ({len(OTHER_SPECIES)} sp.)", _scan(other_files)))

# BirdNET no-bird clips.
if BIRDNET_NO_BIRD_DIR.exists():
    groups.append(
        ("BirdNET no-bird", _scan(sorted(BIRDNET_NO_BIRD_DIR.glob("*.wav"))))
    )

# AudioSet aggregated across focus classes.
if AUDIOSET_DIR.exists():
    as_files: list = []
    for d in sorted(AUDIOSET_DIR.iterdir()):
        if d.is_dir():
            as_files.extend(d.glob("*.wav"))
    groups.append(("AudioSet (all)", _scan(as_files)))

print(
    f"{'group':<32s} {'n':>5s}  {'q05':>6s} {'q50':>6s} {'q95':>6s}  "
    f"{'min':>6s} {'max':>6s}"
)
for label, a in groups:
    if len(a) == 0:
        continue
    print(
        f"{label:<32s} {len(a):>5d}  "
        f"{np.quantile(a, 0.05):>+6.1f} {np.median(a):>+6.1f} {np.quantile(a, 0.95):>+6.1f}  "
        f"{a.min():>+6.1f} {a.max():>+6.1f}"
    )

bins = np.arange(-70, 1, 2)
n = len(groups)
fig, axes = plt.subplots(n, 1, figsize=(10, max(2.0, 1.6 * n)), sharex=True)
if n == 1:
    axes = [axes]
for ax, (label, a) in zip(axes, groups):
    if len(a):
        ax.hist(a, bins=bins, color="steelblue", edgecolor="black", alpha=0.85)
        ax.axvline(
            np.median(a), color="red", lw=1, ls="--", label=f"median {np.median(a):+.1f}"
        )
        ax.legend(loc="upper left", fontsize=8, frameon=False)
    ax.set_ylabel(label, rotation=0, ha="right", va="center")
axes[-1].set_xlabel("RMS (dBFS)")
fig.suptitle("Per-source RMS distribution (all data, pre-split)")
plt.tight_layout()
plt.show()


# Assemble train/val/test

In [ ]:
from building.data.dataset import build_task_dataset

dataset_path = build_task_dataset(
    COLLECTION_NAME,
    target_species=TARGET_SPECIES,
    non_target_species=OTHER_SPECIES,
)
print(dataset_path)